In [1]:
# -------------------------------------------------------------------
# First test: Check if torch is installed and if GPU can be detected
# -------------------------------------------------------------------
# If your Python code uses `torch.use_deterministic_algorithms(True)` then do this before starting Python:
#Sys.setenv(CUBLAS_WORKSPACE_CONFIG = ":4096:8")
#Or just have it at the top of your python script before "import torch"

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch

print(os.environ["CUBLAS_WORKSPACE_CONFIG"])
print(torch.cuda.is_initialized())  

torch.use_deterministic_algorithms(True)

print('torch version:', torch.__version__)
print('torch CUDA build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')

:4096:8
False
torch version: 2.14.1+cpu
torch CUDA build: None
cuda available: False
GPU: no cuda


## Read me

* Purpose: Run VAE-SDE with state-based and Ansari auxilliary Kalman filtering ELBO
* Author: Sy-Miin Chow - coding and final cleaning/code annotations facilitated by ChatGPT-5.5 & Claude 5.0
* Last modified: 10/1/26

All the helper functions are in "HelpfulFuncs_VAE_toshare.py". For a complete listing of the functions and their purposes, see HelpfulFuncs_VAE_function_list.md.

### 1. Getting started!

Import packages, load helper functions, set various random number seeds for reproducibility

In [3]:
from pathlib import Path
import pandas as pd
import json
import numpy as np

from sklearn.model_selection import KFold
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

import importlib
import HelpfulFuncs_VAE_toshare as helpful
importlib.reload(helpful)
from HelpfulFuncs_VAE_toshare import *
import time
import pickle

# ============================================================
# Set device so objects can be put on the right device later
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

#Check my current working directory
from pathlib import Path
print("Current working directory:")
print(Path.cwd())

cuda
Current working directory:
c:\Users\symiin\Dropbox\MachineLearning\Manuscripts\Chow_2024SMEP_Presidential\Revision1\CodetoShare\VAE-SDEDemo\Python


### 2. Define OU SDE parameters arguments

In [4]:
ou_sde_freegamma_param_args = {
    "alpha_value": [-1.0, 1.5],
    "alpha_shift_value": [0.1, -0.1],
    "gamma_shift_X_value": [[0.10, 0.10]],

    # 0 = I_b, 1 = X2_person, 2 = X3_person
    "ib_col": 0,
    "B_x_col": 1,
    "x_cols": 2, #[2],

    "B_value": [
        [-0.3, 0.00],
        [0.00, -0.3],
    ],

    "gamma_B_X2_value": [
        [0.00, 0.00],
        [0.00, 0.00],
    ],
 
    #Parameter bounds
    "parBounds": {
        "B": {
            "lower": [
                [-1.50, -0.75],
                [-0.75, -1.50],
            ],
            "upper": [
                [-0.05, 0.75],
                [0.75, -0.05],
            ],
        },

        "gamma_B_X2": {
            "lower": [
                [0.00, -0.30],
                [-0.30, 0.00],
            ],
            "upper": [
                [0.00, 0.30],
                [0.30, 0.00],
            ],
        },

        "rho": {
            "lower": -0.95,
            "upper": 0.95,
        },
    },

    "rho_value": 0.50,

    "sigma_y_value": [None, 0.60],
    "b_value": [0.0, 0.0],

    "free_alpha": True, #Whether to free set point intercept when Ib=0
    "free_alpha_shift": True, #Whether to free deviation in set point when Ib = 1
    "free_gamma_shift_X": True, #Whether to free coefficient for interaction effect of X3 x Ib on set point
    "free_B": True, #Whether to free elements of the drift matrix
    "free_gamma_B_X2": True, #Whether to free coefficient of X2 on cross-process coupling coefficients in the drift matrix
    "free_rho": True, #Whether to free correlation between the bivariate process noises
    "free_sigma_y": True, #Whether to free measurement error variances
    "free_b": False #Whether to free measurement intercepts. Should always be False as we want interindividual differences in set point to go into the SDEs.
}

# Specify sde argument list of choice
sde_param_args = ou_sde_freegamma_param_args



### 3. Read in pre-generated data set

In [5]:
r = 1
# Data were pre-generated in R to ensure both programs use the exact same data set
sim = read_simData(
    run = r,
    data_file=f"./MCburst_long_run{r}.csv",
    info_file=f"./MCburst_info_run{r}.json",
    device=device
)

globals().update(sim)

seed_num = seed_num = round(1234 + (r-1)/81426)
set_seed(seed_num)

X_train: torch.Size([400, 300, 4])
X_test: torch.Size([100, 300, 4])
n_x: 4
train keys: dict_keys(['Y', 'M', 'times', 'time_mask', 'X', 'id', 'z_true', 'alpha_true', 'p_bin_true', 'mu_cont_true'])


### 3a. Define hyperparameter setting - first, hyperparameters and other setting-related parameters that remain fixed throughout

In [6]:
epochs0 = 100 #Number of epochs for hyperopt optimization
patience0 = 20 #Patience (how many epochs to wait before existing with no improvements) for hyperopt optimization

#Number of hyperopt trials
#Set to 2 for quick demo. Set to 20 in the manuscript to allow the Tree-Structured Parzen (TPE) estimator in Hyperopt to fully kick in.
m_evals0 = 2 

epochs1 = 500 #Number of epochs for final optimization at a fixed set of hyperparameters
patience1 = 50 #Patience for final optimization

isHyperopt = False #Whether to actually do hyperopt optimization
plotHyperopt = False #Whether to plot hyperopt histories

k_folds = 5 #Number of CV folds in hyperparameter optimization

#Ansari's auxiliary filtering-based VAE-SDE
traj_kl_fn=trajectory_auxiliary_kalman_elbo_penalty

#States-based VAE-SDE
traj_kl_fn0 = trajectory_kl_ou_attractor_prior

traj_kl_args={
    "include_logp0": False,
    "transition_mean": "exact",
    "transition_covariance": "euler",
    #"detach_B_i_in_filter": False, #Only affects transition_covariance = "exact"
    "init_state_var": 1.0,
    "initial_state_source": "encoder"
            }

#Setting-related variables that stay fixed throughout.
#Update batch size based on Y_train defined in read_simData()
fixed_train_params = {
    "seed": seed_num,
    "batch_size": int(Y_train.shape[0]/2),
    "max_epochs": epochs1,
    "patience": patience1,
    "k_folds": k_folds,
    "n_layers": 1,
    "weight_decay": 0.00,
    "embedding_hidden_dim": None,
    "eval_use_mean": False
}

k_folds = 5

#Default kl_weight is 0.1. Setting it explicitly for transparency
fixed_train_params.update({'kl_weight': 0.1,#Default kl_weight is 0.1. Setting it explicitly for transparency
                           'seed': seed_num})

out_dir =  f"./HyperoptHistories/"
prefix = f"N{N}Tmax{Tmax}_Tend{Tend}_fold{k_folds}_seed{seed_num}_maxEvals{m_evals0}"

### 3b. Set other hyperparameters - either give fixed values and do optimization (with isHyperopt = TRUE)

In [ ]:
isHyperopt = False #True
# Set time counter
t0 = time.perf_counter()

# Call hyperparameter setting/optimization function
if (isHyperopt == False): #Get default hyperparameter values - fewer arguments needed
    hyper_out = setHyper(
        isHyperopt = isHyperopt,
        fixed_train_params = fixed_train_params,
        Y_train = Y_train,
        M_train = M_train,
        times_train = times_train,
        time_mask_train = time_mask_train,
        X_train_ou = X_train_ou,
        info = info,
        Q = Q,
        traj_kl_fn = traj_kl_fn0, #States-based VAE-SDE; use traj_kl_fn for filtering-based VAE-SDE
        traj_kl_args = traj_kl_args,
        sde_param_args = sde_param_args,
        assign_to_globals=True
    )
else:
    hyper_out = setHyper(
        isHyperopt = isHyperopt,
        fixed_train_params = fixed_train_params,
        Y_train = Y_train,
        M_train = M_train,
        times_train = times_train,
        time_mask_train = time_mask_train,
        X_train_ou = X_train_ou,
        info = info,
        Q = Q,
        traj_kl_fn = traj_kl_fn0, #States-based VAE-SDE; use traj_kl_fn for filtering-based VAE-SDE
        traj_kl_args = traj_kl_args,
        sde_param_args = sde_param_args,
        assign_to_globals=True,
        out_dir = out_dir,
        prefix=prefix,
        m_evals = m_evals0,
        patience0 = patience0,
        epochs0 = epochs0,
        plotHyperopt=True
    )

final_time = time.perf_counter() - t0

best_params = hyper_out['best_params']
print("best_params: ", best_params)
print(f"Hyperparameter optimization over {m_evals0} trials took {final_time/3600:.2f} hour.")

Trial:                                               
0                                                    
Fold:                                                
1                                                    
loss:                                                
459.3087                                             
Y:                                                   
355.1255                                             
KL:                                                  
1041.8324                                            
Trial:                                               
0                                                    
Fold:                                                
2                                                    
loss:                                                
468.1689                                             
Y:                                                   
367.5229                                             
KL:                         

### 4a. [Optional - skipping and using 4b instead] Automated run of VAE-SDE across multiple R_a values (for filtering-based VAE-SDE) and kl_weights

In [ ]:
# res = autoRun(
#     R_a_values   = [nan], #Not needed for states-based VAE-SDE
#     kl_weights   = [1.0, 0.1, 10.0],
#     seeds        = [seed_num],
#     train_data   = train_all,
#     test_arrays  = dict(Y_obs=Y_test, M=M_test, times=times_test,
#                         time_mask=time_mask_test, X=X_test),
#     test_dat     = test,
#     info         = info,
#     Q            = Q,
#     sde_param_args = sde_param_args,
#     trajectory_kl_args= traj_kl_args, #Run filtering approach
#     kl_fn = traj_kl_fn,
#     best_params    = best_params,
#     fixed_train_params = fixed_train_params,
#     max_epochs = epochs1,
#     patience   = patience1,
#     out_csv    = f"Demo_run{r}.csv",
# )


### 4b. Calling fit_exo_sde_vae_fixed_hparams directly

In [9]:
start = time.perf_counter()
final_fit = fit_exo_sde_vae_fixed_hparams(
    params=best_params,
    train_data=train_all,
    valid_data=test_all,
    Q=Q,
    n_x=int(train_all["X"].shape[2]),
    sde_param_args=sde_param_args,
    seed_num=seed_num,
    fixed_train_params=fixed_train_params,
    trajectory_kl_fn=traj_kl_fn,
    trajectory_kl_args = traj_kl_args,
    max_epochs=epochs1,#int(best_params.get("max_epochs", 100)),
    patience=patience1,#int(best_params.get("patience", 30)),
    verbose=True
)

final_model = final_fit["model"]

print("Final test loss:", final_fit["val_loss"])
print("Final test Y:", final_fit["val_y"])
print("Final test KL:", final_fit["val_kl_traj"])
print("Best epoch:", final_fit["best_epoch"])

end = time.perf_counter()

print(f"Elapsed time: {end - start:.2f} seconds")
print(f"Elapsed time: {(end - start) / 60:.2f} minutes")


Using the supplied validation set. Make sure this is NOT the test set.
Epoch    1 | train=1597.2261 train_y=1521.9012 train_kl=753.2487 | val=1088.6123 val_y=1071.2373 val_kl=173.7503
Epoch   10 | train=534.0452 train_y=491.4335 train_kl=426.1167 | val=521.4953 val_y=477.2033 val_kl=442.9201
Epoch   20 | train=441.3957 train_y=399.3532 train_kl=420.4248 | val=447.3749 val_y=406.9701 val_kl=404.0487
Epoch   30 | train=386.2131 train_y=336.3792 train_kl=498.3386 | val=392.6257 val_y=344.8743 val_kl=477.5144
Epoch   40 | train=338.7017 train_y=286.1522 train_kl=525.4956 | val=344.9604 val_y=290.4962 val_kl=544.6420
Epoch   50 | train=294.5324 train_y=232.6179 train_kl=619.1450 | val=299.2287 val_y=236.5959 val_kl=626.3282
Epoch   60 | train=249.0560 train_y=175.7504 train_kl=733.0564 | val=252.7738 val_y=178.2414 val_kl=745.3240
Epoch   70 | train=205.2387 train_y=116.3573 train_kl=888.8145 | val=208.2760 val_y=127.2882 val_kl=809.8773
Epoch   80 | train=149.4106 train_y=56.1446 train_kl=

### 5. Check results and export to R

In [10]:
est_test = extract_python_exo_sde_estimates(
    model=final_fit["model"],
    Y_obs=Y_test,
    M=M_test,
    times=times_test,
    time_mask=time_mask_test,
    X=X_test,
    use_mean=True
)

print(est_test.keys())

# Save for inspection in R for comparisons
save_python_estimates_for_r_with_fingerprint(
    est=est_test,
    filename="./SavedPythonVAE_SDE_estimates.npz",
    id_test=test["id"],          # or whatever vector defines test row order
    times_test=test["times"],
    time_mask_test=test["time_mask"],
    z_true_test=test["z_true"]
)

#Testing to see if the saved file is correct
filename = "./SavedPythonVAE_SDE_estimates.npz"
npz = np.load(filename)
print(npz.files)

dict_keys(['z_hat', 'z_draw_hat', 'mu_y_hat', 'Sigma_q', 'z_se', 'A_hat', 'b_hat', 'sigma_y_hat', 'sde_hat', 'z_filt_mean_hat', 'z_filt_cov_hat', 'z_filt_se_hat', 'z_pred_mean_hat', 'z_pred_cov_hat', 'z_pred_se_hat', 'z_smooth_mean_hat', 'z_smooth_cov_hat', 'z_smooth_se_hat', 'alpha_t_hat', 'B_i_filter_hat', 'logp_weight', 'log_pa_hat', 'log_qa_given_y_hat', 'aux_kl_hat'])
['z_hat', 'z_draw_hat', 'mu_y_hat', 'Sigma_q', 'z_se', 'A_hat', 'b_hat', 'sigma_y_hat', 'sde_hat__alpha', 'sde_hat__alpha_shift', 'sde_hat__gamma_shift_X', 'sde_hat__B', 'sde_hat__rho', 'sde_hat__R_a', 'sde_hat__gamma_B_X2', 'sde_hat__B_x_col', 'sde_hat__use_gamma_B_X2', 'sde_hat__free_gamma_B_X2', 'sde_hat__ib_col', 'sde_hat__x_cols', 'sde_hat__use_X_effects', 'sde_hat__free_gamma_shift_X', 'sde_hat__use_amortized_embedding', 'sde_hat__embedding_to_alpha_shift', 'sde_hat__embedding_to_B_offdiag', 'z_filt_mean_hat', 'z_filt_cov_hat', 'z_filt_se_hat', 'z_pred_mean_hat', 'z_pred_cov_hat', 'z_pred_se_hat', 'z_smooth_mea